# Lab 5: 图像风格迁移 — Qwen3-VL + Z-Image-Turbo

本实验结合 **Lab1（Qwen3-VL 视觉理解）** 和 **Lab4（Z-Image-Turbo 图像生成）**，实现一个创新的图像风格迁移应用。

**核心思路**：
1. 用户上传一张**风格参考图**
2. VLM（Qwen3-VL）分析该图片，提取风格特征（色彩、氛围、构图、艺术风格等）
3. 用户输入**内容描述**
4. 图像生成模型（Z-Image-Turbo）根据"内容描述 + VLM提取的风格特征"生成新图

**创新点**：
- 基于视觉语言模型的智能风格分析，无需人工设计特征
- 端侧部署，使用 OpenVINO 在 Intel AI PC 上高效运行

#### 目录：
- [下载模型](#下载模型)
- [加载模型](#加载模型)
- [风格分析测试](#风格分析测试)
- [图像生成测试](#图像生成测试)
- [交互式演示](#交互式演示)

## 下载模型
[返回目录 ⬆️](#目录：)

从 ModelScope 下载 Qwen3-VL-4B OpenVINO INT4 模型和 Z-Image-Turbo 模型。

In [ ]:
from pathlib import Path

# 模型存放路径 (放在项目根目录)
project_root = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
models_dir = project_root.parent  # modelscope-contest

# Qwen3-VL 模型
vlm_model_dir = models_dir / "Qwen3-VL-4B-Instruct-int4-ov"
if not vlm_model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-VL-4B-Instruct-int4-ov", local_dir=str(vlm_model_dir))
    print(f"VLM 模型已下载到: {vlm_model_dir}")
else:
    print(f"VLM 模型已存在: {vlm_model_dir}，跳过下载")

# Z-Image-Turbo 模型
image_model_dir = models_dir / "Z-Image-Turbo-int4-ov"
if not image_model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("ZhipuAI/ImageGen-Turbo-Instruct-int4-ov", local_dir=str(image_model_dir))
    print(f"图像生成模型已下载到: {image_model_dir}")
else:
    print(f"图像生成模型已存在: {image_model_dir}，跳过下载")

## 加载模型
[返回目录 ⬆️](#目录：)

In [ ]:
# Patch Qwen3-VL config 动态添加 embed_dim 字段
import json
from pathlib import Path

vlm_model_dir = Path("../Qwen3-VL-4B-Instruct-int4-ov")
if not vlm_model_dir.exists():
    vlm_model_dir = Path("Qwen3-VL-4B-Instruct-int4-ov")

with open(vlm_model_dir / "config.json") as f:
    config = json.load(f)

# 兼容处理：确保 vision_config 包含 embed_dim
if 'vision_config' in config and 'embed_dim' not in config['vision_config']:
    config['vision_config']['embed_dim'] = config['vision_config']['hidden_size']
    with open(vlm_model_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    print("已修补 config.json，添加 embed_dim 字段")
else:
    print("config.json 无需修补或已包含 embed_dim")

In [ ]:
# 加载 VLM 模型
from optimum.intel.openvino.modeling_visual_language import MODEL_TYPE_TO_CLS_MAPPING, _OVQwen2VLForCausalLM
from optimum.intel.openvino import OVModelForVisualCausalLM
from transformers import AutoProcessor

# 关键：Monkey Patch - 注册 Qwen3-VL 模型类型
MODEL_TYPE_TO_CLS_MAPPING['qwen3_vl'] = _OVQwen2VLForCausalLM

print("正在加载 Qwen3-VL 模型...")
vlm_model = OVModelForVisualCausalLM.from_pretrained(vlm_model_dir, device='CPU')

# 配置 processor
vlm_processor = AutoProcessor.from_pretrained(
    vlm_model_dir,
    min_pixels=256*28*28,
    max_pixels=1280*28*28,
    fix_mistral_regex=True
)
print("Qwen3-VL 模型加载完成！")

In [ ]:
# 加载 Z-Image-Turbo 图像生成模型
from z_image_turbo_ov import ZImageTurboOV

image_model_dir = Path("../Z-Image-Turbo-int4-ov")
if not image_model_dir.exists():
    image_model_dir = Path("Z-Image-Turbo-int4-ov")

print("正在加载 Z-Image-Turbo 模型...")
image_generator = ZImageTurboOV(str(image_model_dir), device='CPU')
print("Z-Image-Turbo 模型加载完成！")

## 风格分析测试
[返回目录 ⬆️](#目录：)

使用 Qwen3-VL 分析风格参考图，提取结构化的艺术风格描述。

In [ ]:
from PIL import Image
import requests
from pathlib import Path

# 下载示例风格图片
style_image_url = "https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/notebooks/yy.jpg"
style_image_path = Path("style_reference.jpg")

if not style_image_path.exists():
    Image.open(requests.get(style_image_url, stream=True).raw).save(style_image_path)
    print(f"风格参考图已下载: {style_image_path}")
else:
    print(f"风格参考图已存在: {style_image_path}")

# 显示风格参考图
style_image = Image.open(style_image_path)
display(style_image.resize((512, 512)))

In [ ]:
from gradio_helper import analyze_style

# 使用 Gradio Helper 中的风格分析函数（流式输出）
print("正在分析风格，请稍候...")
print("-" * 50)

style_description = None
for result in analyze_style(vlm_model, vlm_processor, str(style_image_path)):
    print(result, end="\r")
    style_description = result

print("\n" + "-" * 50)
print(f"\n风格分析完成！")
print(f"风格描述: {style_description}")

## 图像生成测试
[返回目录 ⬆️](#目录：)

使用 Z-Image-Turbo 根据内容描述和风格描述生成图像。

In [ ]:
import numpy as np

# 定义内容描述
content_prompt = "一位优雅的中国古典美女"

# 结合风格描述（使用前一步的分析结果或手动定义）
style_description = style_description or """古典东方美学风格，浓郁的中国传统色彩（红、金、黑），工笔画细腻线条，仕女画优雅仪态，丝帛质感，古典家具背景，宁静雅致的氛围。"""

# 拼接最终 prompt
final_prompt = f"{content_prompt}，{style_description}"

print(f"内容描述: {content_prompt}")
print(f"风格描述: {style_description[:50]}...")
print(f"最终 Prompt: {final_prompt[:80]}...")

In [ ]:
# 生成图像
# num_inference_steps: 步数越多质量越高（9步快速预览，100步高质量）
print("正在生成图像...")
image = image_generator.generate(
    prompt=final_prompt,
    height=512,
    width=512,
    num_inference_steps=100,  # 高质量输出
    seed=int(np.random.randint(0, 1000000))
)

display(image)
print("图像生成完成！")

## 交互式演示
[返回目录 ⬆️](#目录：)

通过 Gradio 界面体验完整的图像风格迁移功能：
1. 上传风格参考图
2. 系统自动分析风格特征
3. 输入内容描述
4. 生成并展示风格化图像

**注意**：运行此单元格后，会启动 Gradio Web 服务，请在输出的 URL 访问界面。

In [ ]:
from gradio_helper import make_demo

# 创建并启动 Gradio 演示
print("启动 Gradio 演示...")
demo = make_demo(
    vlm_model=vlm_model,
    vlm_processor=vlm_processor,
    image_generator=image_generator,
    model_name="Style Transfer"
)

# 启动服务
demo.launch(debug=True, share=True, server_name="0.0.0.0")